In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torch.utils.data import ConcatDataset, DataLoader

import sys
import os

current_dir = os.path.dirname(os.path.abspath(__file__))
sys.path.append(current_dir)

from utils.EEDDataLoader import EEGDataset
from utils.train_eval import eval, train
from models.ConvParallelEEG1DModel import ConvParallelEEG1DModel

files_list_csv = "./dataset/meta_data.csv"
train_data_csv = "./dataset/train.csv"
val_data_csv = "./dataset/eval.csv"
test_data_csv = "./dataset/test.csv"
prediction_saving_csv_train = "./dataset/predictions/predictions_train_conv_crops.csv"
prediction_saving_csv_eval = "./dataset/predictions/predictions_eval_conv_crops.csv"
prediction_saving_csv_test = "./dataset/predictions/predictions_test_conv_crops.csv"
model_saving_name = "./dataset/model/weights/conv_tt_0_original.pt"

epochs = 50
batch_size = 64
hidden_size = 20
output_size = 3
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [0]


In [ ]:
train_dataset = EEGDataset(train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(test_data_csv, number_of_eeg_channels=number_of_eeg_channels)
combined_dataset = ConcatDataset([train_dataset, val_dataset])

# Split the data into training and validation sets
train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [ ]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = ConvParallelEEG1DModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
prev_eval_loss = 0
prev_eval_acc = 0

In [ ]:
model.eval()
# total trainable params
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {params:,}  (~{params*4/1e6:.2f} MB in FP32)")  # 4 bytes/param

In [ ]:
model = model.to(device)

prev_eval_loss, prev_eval_acc = eval(model, val_loader, loss_fn, 1,False,'', active_branches, device)
print("Loss:",prev_eval_loss, "Accuracy",prev_eval_acc)

In [ ]:
result = {"epoch":[],"train_loss":[],"train_acc":[],"eval_loss":[],"eval_acc":[],"test_acc":[],"test_loss":[]}
saved_epoch = 0

In [ ]:
for epoch in range(epochs):  # loop over the dataset multiple times
    train_loss, train_acc = train(model, train_loader,optimizer, loss_fn, active_branches, device)
    print(f"Train Loss after epoch {epoch+2}: {train_loss} Train ACC after epoch {epoch}: {train_acc}")
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    print(f"VAL Loss after epoch {epoch}: {eval_loss} VAL ACC after epoch {epoch}: {eval_acc}")
    # test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    test_loss = eval_loss
    test_acc = eval_acc
    print(f"Test Loss after epoch {epoch}: {test_loss} Test Acc after epoch {epoch}: {test_acc}")

    scheduler.step(eval_loss)
    
    result['epoch'].append(epoch)
    
    result['train_loss'].append(train_loss)
    result['train_acc'].append(train_acc)
    result['eval_loss'].append(eval_loss)
    result['eval_acc'].append(eval_acc)
    result['test_loss'].append(test_loss)
    result['test_acc'].append(test_acc)

    if prev_eval_acc < eval_acc:
      prev_eval_acc = eval_acc
      saved_epoch = epoch
      torch.save(model.state_dict(), model_saving_name)
      print("WEIGHTS SAVED")

In [ ]:
print("Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  ")
for epoch in range(epochs):
    if(saved_epoch == epoch):
        print("Saved Weights")
    print(result['epoch'][epoch],"|",result['train_loss'][epoch],"|",result['train_acc'][epoch],"|",result['eval_loss'][epoch],"|",result['eval_acc'][epoch],"|",result['test_loss'][epoch],"|",result['test_acc'][epoch])

In [ ]:
model_state_dict = torch.load(model_saving_name)
# Load the state_dict into the model
model.load_state_dict(model_state_dict)
eval_loss, eval_acc = eval(model, val_loader, loss_fn, eval_batch,True,prediction_saving_csv_eval, active_branches, device)
print(f"VAL Loss: {eval_loss} VAL ACC: {eval_acc}")